# NB01 — Ekstraksi Fitur Wajah

**Tujuan:** Mendeteksi dan mengekstraksi face embedding dari dataset foto baru  
**Model:** InsightFace `buffalo_l` → embedding 512-dimensi  
**Output:** `embeddings.npy`, `metadata.pkl`  

---

**Alur:**
```
Foto (JPG/PNG/...) 
  → InsightFace buffalo_l 
  → deteksi bbox + ekstraksi embedding (512-dim)
  → simpan ke file
```

## 0. Instalasi & Import

In [ ]:
# Install jika belum tersedia
# !pip install insightface onnxruntime opencv-python pillow tqdm

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import cv2
import insightface
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from insightface.app import FaceAnalysis
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print(f"InsightFace version : {insightface.__version__}")
print(f"NumPy version       : {np.__version__}")

## 1. Konfigurasi

In [ ]:
# ─── SESUAIKAN PATH INI ──────────────────────────────────────────────────────
DATA_DIR    = Path("../../data/dataset_baru")   # folder foto dataset
OUTPUT_DIR  = Path("./output")                  # hasil ekstraksi disimpan di sini
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Format foto yang didukung
SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".tif"}

# InsightFace config
DET_SIZE   = (640, 640)   # ukuran input detection
DET_THRESH = 0.5          # threshold deteksi wajah
CTX_ID     = 0            # 0 = GPU, -1 = CPU

print(f"Dataset dir : {DATA_DIR.resolve()}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")

## 2. Load Daftar Foto

In [ ]:
all_photos = sorted([
    p for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in SUPPORTED_EXT
])

print(f"Total foto ditemukan : {len(all_photos)}")
print(f"\nContoh 5 foto pertama:")
for p in all_photos[:5]:
    print(f"  {p.name}")

## 3. Inisialisasi Model InsightFace

In [ ]:
face_app = FaceAnalysis(
    name="buffalo_l",
    allowed_modules=["detection", "recognition"],
)
face_app.prepare(ctx_id=CTX_ID, det_size=DET_SIZE, det_thresh=DET_THRESH)

print("Model buffalo_l berhasil dimuat.")
print("Dimensi embedding : 512")

## 4. Ekstraksi Embedding

> Satu foto bisa menghasilkan **beberapa** embedding jika ada lebih dari satu wajah.

In [ ]:
embeddings_list = []   # List[np.array(512,)]
metadata_list   = []   # List[dict]

stats = {
    "total_photos"    : len(all_photos),
    "photos_with_face": 0,
    "photos_no_face"  : 0,
    "photos_error"    : 0,
    "total_faces"     : 0,
}

for photo_path in tqdm(all_photos, desc="Ekstraksi wajah"):
    try:
        img = cv2.imread(str(photo_path))
        if img is None:
            # Coba via PIL untuk format non-standar
            pil_img = Image.open(photo_path).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

        faces = face_app.get(img)

        if len(faces) == 0:
            stats["photos_no_face"] += 1
            continue

        stats["photos_with_face"] += 1
        stats["total_faces"]      += len(faces)

        for face_idx, face in enumerate(faces):
            emb = face.normed_embedding   # sudah L2-normalized, shape (512,)
            embeddings_list.append(emb)
            metadata_list.append({
                "photo_path" : str(photo_path),
                "photo_name" : photo_path.name,
                "face_idx"   : face_idx,
                "bbox"       : face.bbox.tolist(),       # [x1,y1,x2,y2]
                "det_score"  : float(face.det_score),
            })

    except Exception as e:
        stats["photos_error"] += 1
        print(f"[ERROR] {photo_path.name}: {e}")

print("\nEkstraksi selesai!")
print(f"  Foto total          : {stats['total_photos']}")
print(f"  Foto ada wajah      : {stats['photos_with_face']}")
print(f"  Foto tanpa wajah    : {stats['photos_no_face']}")
print(f"  Foto error          : {stats['photos_error']}")
print(f"  Total wajah (face)  : {stats['total_faces']}")

## 5. Konversi ke Array & Verifikasi

In [ ]:
embeddings = np.array(embeddings_list)   # shape: (N_faces, 512)

print(f"Shape embeddings : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")

# Verifikasi normalisasi L2
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nVerifikasi L2-norm (harusnya ≈ 1.0):")
print(f"  Min  : {norms.min():.6f}")
print(f"  Max  : {norms.max():.6f}")
print(f"  Mean : {norms.mean():.6f}")

# Rata-rata wajah per foto
n_photos_with_face = stats['photos_with_face']
avg_faces = stats['total_faces'] / n_photos_with_face if n_photos_with_face > 0 else 0
print(f"\nRata-rata wajah per foto : {avg_faces:.2f}")

## 6. Simpan Output

In [ ]:
emb_path  = OUTPUT_DIR / "embeddings.npy"
meta_path = OUTPUT_DIR / "metadata.pkl"

np.save(emb_path, embeddings)
with open(meta_path, "wb") as f:
    pickle.dump(metadata_list, f)

print(f"Tersimpan:")
print(f"  {emb_path}  ({emb_path.stat().st_size / 1024:.1f} KB)")
print(f"  {meta_path}  ({meta_path.stat().st_size / 1024:.1f} KB)")

## 7. Analisis & Visualisasi

In [ ]:
# ── 7a. Distribusi jumlah wajah per foto ──────────────────────────────────────
from collections import Counter

faces_per_photo = Counter(m["photo_name"] for m in metadata_list)
face_counts = list(faces_per_photo.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(face_counts, bins=range(1, max(face_counts) + 2), color="#4F46E5",
             edgecolor="white", align="left")
axes[0].set_xlabel("Jumlah wajah per foto", fontsize=11)
axes[0].set_ylabel("Jumlah foto", fontsize=11)
axes[0].set_title("Distribusi Wajah per Foto", fontsize=13, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Pie chart: foto dengan wajah vs tidak
pie_data   = [stats["photos_with_face"], stats["photos_no_face"], stats["photos_error"]]
pie_labels = ["Ada wajah", "Tanpa wajah", "Error"]
pie_colors = ["#4F46E5", "#94A3B8", "#F87171"]
axes[1].pie(
    [v for v in pie_data if v > 0],
    labels=[l for v, l in zip(pie_data, pie_labels) if v > 0],
    colors=[c for v, c in zip(pie_data, pie_colors) if v > 0],
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 10}
)
axes[1].set_title("Komposisi Dataset", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribusi_wajah.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nStatistik jumlah wajah per foto:")
print(f"  Min  : {min(face_counts)}")
print(f"  Max  : {max(face_counts)}")
print(f"  Mean : {np.mean(face_counts):.2f}")
print(f"  Std  : {np.std(face_counts):.2f}")

In [ ]:
# ── 7b. Distribusi detection score ────────────────────────────────────────────
det_scores = [m["det_score"] for m in metadata_list]

plt.figure(figsize=(8, 3.5))
plt.hist(det_scores, bins=40, color="#06B6D4", edgecolor="white")
plt.axvline(np.mean(det_scores), color="#F59E0B", linewidth=2,
            label=f"Mean = {np.mean(det_scores):.3f}")
plt.xlabel("Detection Score", fontsize=11)
plt.ylabel("Frekuensi", fontsize=11)
plt.title("Distribusi Detection Score Wajah", fontsize=13, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "det_score_dist.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDetection score:")
print(f"  Min  : {min(det_scores):.4f}")
print(f"  Max  : {max(det_scores):.4f}")
print(f"  Mean : {np.mean(det_scores):.4f}")

In [ ]:
# ── 7c. Preview: sample face crops ────────────────────────────────────────────
def crop_face(photo_path, bbox, padding=0.2):
    """Crop wajah dari foto dengan sedikit padding."""
    img = Image.open(photo_path).convert("RGB")
    W, H = img.size
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, x1 - bw * padding)
    y1 = max(0, y1 - bh * padding)
    x2 = min(W, x2 + bw * padding)
    y2 = min(H, y2 + bh * padding)
    return img.crop((x1, y1, x2, y2)).resize((112, 112))

np.random.seed(42)
sample_idx = np.random.choice(len(metadata_list), min(20, len(metadata_list)), replace=False)

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle("Sample Face Crops (random 20)", fontsize=13, fontweight="bold")

for i, idx in enumerate(sample_idx):
    ax = axes[i // 10, i % 10]
    try:
        m   = metadata_list[idx]
        img = crop_face(m["photo_path"], m["bbox"])
        ax.imshow(img)
        ax.set_title(f"{m['det_score']:.2f}", fontsize=7)
    except Exception:
        ax.text(0.5, 0.5, "err", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_crops.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Ringkasan

In [ ]:
coverage_pct = stats['photos_with_face'] / stats['total_photos'] * 100

print("=" * 50)
print("  RINGKASAN NB01 — EKSTRAKSI FITUR")
print("=" * 50)
print(f"  Total foto input     : {stats['total_photos']:,}")
print(f"  Foto ada wajah       : {stats['photos_with_face']:,}  ({coverage_pct:.1f}%)")
print(f"  Foto tanpa wajah     : {stats['photos_no_face']:,}")
print(f"  Total face embedding : {len(embeddings_list):,}")
print(f"  Dimensi embedding    : {embeddings.shape[1]}")
print(f"  L2-normalized        : Ya (normed_embedding InsightFace)")
print("=" * 50)
print(f"  Output:")
print(f"    embeddings.npy  → {embeddings.shape}")
print(f"    metadata.pkl    → {len(metadata_list)} records")
print("=" * 50)
print()
print("➜  Lanjut ke NB02_EDA_Baseline.ipynb")